## 03 · Trigger initial sync and wait READY

Pre-requisites (must be complete before running this notebook):
- `01_symptom_corpus.py` — `workspace.default.symptom_corpus` exists, CDF=true
- `02_vs_endpoint.py` — VS endpoint online, Delta-sync index created (TRIGGERED)

**No agent code may run until `INDEX_READY = True` at the bottom of this notebook.**

In [1]:
import json
import os
import subprocess
import sys
import time

PROFILE    = os.environ.get("DBX_PROFILE", "tero2")
INDEX_NAME = "workspace.default.symptom_corpus_idx"

READY_STATES   = {"ONLINE", "ONLINE_NO_PENDING_UPDATE"}
POLL_INTERVAL  = 15   # seconds between status checks
POLL_MAX       = 80   # max iterations ≈ 20 min

print(f"Index : {INDEX_NAME}")
print(f"Profile: {PROFILE}")

Index : workspace.default.symptom_corpus_idx
Profile: tero2


In [2]:
def api(method: str, path: str, body=None) -> dict:
    """Thin wrapper around `databricks api`; tolerates non-zero exit when stdout
    is valid JSON (VS API returns 4xx on benign cases like already-syncing)."""
    cmd = ["databricks", "api", method, "-p", PROFILE, path]
    if body is not None:
        cmd += ["--json", json.dumps(body)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = r.stdout or ""
    if r.returncode != 0 and not out.strip():
        raise RuntimeError(
            f"api({method} {path}) transport error rc={r.returncode}: "
            f"{(r.stderr or '').strip()[:300]}"
        )
    if not out.strip():
        return {}
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        raise RuntimeError(f"api({method} {path}) non-JSON response: {out[:300]}")

In [3]:
# ── Trigger initial sync ──────────────────────────────────────────────────────
# TRIGGERED pipeline type does NOT auto-sync on table write; an explicit POST
# /sync is required.  We tolerate HTTP 4xx (already syncing) gracefully.

print(f"→ triggering sync on {INDEX_NAME!r}...")
sync_resp = api("post", f"/api/2.0/vector-search/indexes/{INDEX_NAME}/sync")
print(f"  sync response: {sync_resp or '(empty — likely already syncing, that is OK)'}")

→ triggering sync on 'workspace.default.symptom_corpus_idx'...
  sync response: (empty — likely already syncing, that is OK)


In [4]:
# ── Poll until READY ──────────────────────────────────────────────────────────
# READY = detailed_state in {ONLINE, ONLINE_NO_PENDING_UPDATE}
# No agent code runs until INDEX_READY is True.

INDEX_READY = False

print(f"→ polling index state (up to {POLL_MAX * POLL_INTERVAL // 60} min)...")
for i in range(POLL_MAX):
    idx = api("get", f"/api/2.0/vector-search/indexes/{INDEX_NAME}")
    status        = idx.get("status", {})
    detail_state  = status.get("detailed_state", "?")
    indexed_rows  = status.get("indexed_row_count", 0)
    message       = status.get("message", "")

    print(f"  [{i * POLL_INTERVAL:>4}s] state={detail_state:<35} indexed={indexed_rows}")

    if detail_state in READY_STATES:
        INDEX_READY = True
        break

    if "FAILED" in detail_state:
        raise RuntimeError(
            f"Index entered failure state {detail_state!r}: {message or idx}"
        )

    time.sleep(POLL_INTERVAL)

if not INDEX_READY:
    raise RuntimeError(
        f"Index {INDEX_NAME!r} did not reach READY within "
        f"{POLL_MAX * POLL_INTERVAL // 60} min.  "
        "Last state: " + detail_state
    )

→ polling index state (up to 20 min)...
  [   0s] state=ONLINE_UPDATING_PIPELINE_RESOURCES     indexed=100
  [  15s] state=ONLINE_NO_PENDING_UPDATE               indexed=100


In [5]:
# ── READY gate ────────────────────────────────────────────────────────────────
# This assertion is the hard boundary — downstream agent code MUST NOT be
# placed above this cell.  If INDEX_READY is False (timeout or failure),
# the RuntimeError above already aborted the notebook.

assert INDEX_READY, (
    f"INDEX_READY={INDEX_READY!r} — "
    "do not proceed with agent code until the index is READY."
)

print(f"\n✓ {INDEX_NAME} is READY")
print(f"  state         = {detail_state}")
print(f"  indexed_rows  = {indexed_rows}")
print("\n  Agent code is now safe to run.")


✓ workspace.default.symptom_corpus_idx is READY
  state         = ONLINE_NO_PENDING_UPDATE
  indexed_rows  = 100

  Agent code is now safe to run.


In [ ]:
# ── Block 10 — Smoke Vector Search query ─────────────────────────────────────
# MUST include red_flags in columns:
#   Block 11 (triage.py) reads top[4] as red_flags.
#   Omitting it here makes the smoke pass (4 columns returned) but the agent
#   crashes with IndexError on top[4] at runtime.
#
# Expected: top hit for "chest pain sweating" is sx_3 (EN) or sx_4 (HI),
#   both specialty=Cardiology, urgency=5, red_flags=["chest_pain","sweating"].

ENDPOINT  = os.environ.get("VS_ENDPOINT", "mubarak_vs")
QUERY     = "chest pain sweating left arm pain"
COLUMNS   = ["id", "text", "specialty", "urgency", "red_flags"]  # must keep red_flags at index 4
NUM       = 3

print(f"→ Block 10 smoke query: {QUERY!r}")
print(f"  endpoint={ENDPOINT}  index={INDEX_NAME}  columns={COLUMNS}")

r = api(
    "post",
    f"/api/2.0/vector-search/indexes/{INDEX_NAME}/query",
    {
        "query_text": QUERY,
        "num_results": NUM,
        "columns": COLUMNS,
    },
)
hits = r.get("result", {}).get("data_array", [])

assert hits, (
    "VS query returned no results — index may be empty or not yet synced. "
    "Re-run 01_symptom_corpus.py to repopulate, then trigger sync and wait READY again."
)

print(f"\n  {len(hits)} hit(s):")
for row in hits:
    _id, _text, specialty, urgency, red_flags = row[0], row[1], row[2], row[3], row[4]
    print(f"    id={_id:<8}  specialty={specialty:<22}  urgency={urgency}  red_flags={red_flags}")

top_id = hits[0][0]
assert top_id in ("sx_3", "sx_4"), (
    f"Expected top hit to be sx_3 or sx_4 (cardiology); got {top_id!r}. "
    "The sx_ cardiology entries may not yet be in the index — re-run 01_symptom_corpus.py, "
    "trigger sync (cell 3), wait READY (cell 4), then re-run this cell."
)

top_red_flags = hits[0][4]
assert top_red_flags is not None, (
    f"top[4] (red_flags) is None for {top_id!r} — "
    "red_flags column missing from the Delta table. Re-run 01_symptom_corpus.py."
)

print(f"\n✓ Block 10 smoke passed")
print(f"  top hit : {top_id}  specialty={hits[0][2]}  urgency={hits[0][3]}")
print(f"  red_flags column present at index 4: {top_red_flags}")
print("  triage.py top[4] is safe (no IndexError).")
